In [1]:
!pip install mysql-connector-python pandas


In [2]:
import pandas as pd
import mysql.connector
from mysql.connector import errorcode

CSV_PATH = r"C:\\Users\\tejas\\Downloads\\event_project_sql\\event_attendance.csv"  # change this
CHUNK_SIZE = 10_000

DB_CONFIG = {
    "host": "127.0.0.1",
    "user": "root",
    "password": "stacy@180702",
    "database": "event_management",   # change this
    "port": 3306,
}

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS stg_event_attendance (
  event_id INT NOT NULL,
  event_name VARCHAR(255) NOT NULL,
  location VARCHAR(255) NOT NULL,
  date_time DATETIME NOT NULL,
  attendee_name VARCHAR(200) NOT NULL,
  attendee_email VARCHAR(255) NOT NULL,
  attendee_phone_number VARCHAR(50),
  INDEX idx_stg_event_id (event_id),
  INDEX idx_stg_attendee_email (attendee_email)
);
"""

INSERT_SQL = """
INSERT INTO stg_event_attendance
(event_id, event_name, location, date_time, attendee_name, attendee_email, attendee_phone_number)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

def main():
    cnx = mysql.connector.connect(**DB_CONFIG)
    cur = cnx.cursor()

    # Create table
    cur.execute(CREATE_TABLE_SQL)
    cnx.commit()

    total = 0

    # Read CSV in chunks and insert
    for df in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
        # Ensure column order matches INSERT_SQL exactly
        df = df[
            ["event_id", "event_name", "location", "date_time",
             "attendee_name", "attendee_email", "attendee_phone_number"]
        ]

        # Convert NaN -> None (MySQL NULL)
        df = df.where(pd.notnull(df), None)

        rows = [tuple(x) for x in df.to_numpy()]
        cur.executemany(INSERT_SQL, rows)  # batch insert [web:159]
        cnx.commit()

        total += len(rows)
        print(f"Inserted {total:,} rows...")

    cur.close()
    cnx.close()
    print("Done.")

if __name__ == "__main__":
    main()


Inserted 10,000 rows...
Inserted 20,000 rows...
Inserted 30,000 rows...
Inserted 40,000 rows...
Inserted 50,000 rows...
Inserted 60,000 rows...
Inserted 70,000 rows...
Inserted 80,000 rows...
Inserted 90,000 rows...
Inserted 100,000 rows...
Inserted 110,000 rows...
Inserted 120,000 rows...
Inserted 130,000 rows...
Inserted 140,000 rows...
Inserted 150,000 rows...
Inserted 160,000 rows...
Inserted 170,000 rows...
Inserted 180,000 rows...
Inserted 190,000 rows...
Inserted 200,000 rows...
Done.
